# IFEST DAC 2026 — V4 Transductive Canonical Matching

**Agreed V4 design only. No new Transformer training.**

V4 keeps the already-submitted **V2 system (public LB 0.94856) as the frozen base** and adds only the transductive test–test structural inference supported by the previous audit:

1. Preserve every V2 train→test graph-forced decision exactly.
2. On unresolved test rows sharing **exact normalized content but different titles**, use one-canonical-positive matching only when the content is not malformed, no positive title is known from train, and the best candidate rank is at least `0.20`.
3. On still-unresolved rows sharing **exact normalized title but different contents**, apply the smaller extension only when no positive content is known from train, best rank is at least `0.90`, and top-vs-second margin is at least `0.02`.
4. Candidate ranking is frozen as `0.45 × V3 blend probability + 0.55 × V2 global entailment`.
5. Otherwise keep the original V2 final label.

There is **no fuzzy matching, no model retraining, no new backbone, no leaderboard-tuned manual correction, and no modification of the V2 graph package**. The notebook embeds the frozen V2/V3 score artifacts so it can run directly from the competition dataset in Kaggle.

Validation generated by the notebook is diagnostic for this **transductive inference mode**: train structural profiling, component-disjoint transductive folds, and 100 repeated pseudo-competition splits. These diagnostics do not train or retune the frozen V2/V3 models.

In [5]:
import os, re, io, json, gzip, base64, hashlib, unicodedata, warnings, zipfile
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedGroupKFold, train_test_split

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 180)

# ---------------- V4 FROZEN CONFIG ----------------
SEED = 2026
N_FOLDS = 3
N_PSEUDO_SPLITS = 100

# Frozen V2 graph package.
V2_TITLE_GATE = 0.9839511586057019
MIN_GRAPH_CONTENT_WORDS = 20

# Frozen V4 transductive package.
RANK_V3_WEIGHT = 0.45
RANK_V2_GLOBAL_WEIGHT = 0.55
CONTENT_CANONICAL_MIN_SCORE = 0.20
TITLE_CANONICAL_MIN_SCORE = 0.90
TITLE_CANONICAL_MIN_MARGIN = 0.02

PUBLIC_LB_V1 = 0.93367
PUBLIC_LB_V2 = 0.94856
PUBLIC_LB_V3 = 0.94856

EXPECTED_TRAIN_ROWS = 14397
EXPECTED_TEST_ROWS = 3603
# Current official Kaggle CSV mirror deterministically yields 16 V4 changes.
EXPECTED_V4_CHANGES_VS_V2 = 16
EXPECTED_DATA_HASHES = {
    "train": "d2c3cf75cb6ef8d502f32d9b8ed8b22508040ea8570a295269a53b67e1e184ab",
    "test": "0871476c2c74ee5ba6daa931dbd6cfe1d3a992ac3da9c4ea5228c1deed16cf88",
    "sample_submission": "2725f8fe1c074b5e1d7c7f1234cfa482871d2ea94f6056b4451c22552809ebf5",
}

OUTPUT_DIR = Path("/kaggle/working" if Path("/kaggle/working").exists() else "/mnt/data/ifest_v4_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("IFEST DAC V4 — Transductive Canonical Matching")
print("Training: intentionally disabled by design")
print(f"Rank = {RANK_V3_WEIGHT:.2f}*V3_blend + {RANK_V2_GLOBAL_WEIGHT:.2f}*V2_global_entailment")
try:
    import torch
    print(f"PyTorch={torch.__version__} | CUDA={torch.cuda.is_available()} | GPUs={torch.cuda.device_count()}")
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            print(i, torch.cuda.get_device_name(i))
except Exception as e:
    print("Torch diagnostic unavailable:", repr(e))

IFEST DAC V4 — Transductive Canonical Matching
Training: intentionally disabled by design
Rank = 0.45*V3_blend + 0.55*V2_global_entailment
PyTorch=2.10.0+cu128 | CUDA=True | GPUs=2
0 Tesla T4
1 Tesla T4


In [6]:
FROZEN_ARTIFACTS = {
    'V2_OOF': {
        'sha256': 'ed13cfd3136181a87a087a5ef9276bd78248c707ff96509ab896800919010a5e',
        'rows': 14397,
        'columns': ['id', 'label', 'meta_prob', 'global_e'],
        'payload': (
            '...'
        ),
    },
    'V2_TEST': {
        'sha256': '5f52f3664cf826511f8dfaf0a8ffbb4a13917ecbb4af7432b56a3240155da7b5',
        'rows': 3603,
        'columns': ['id', 'meta_prob', 'global_e', 'graph_rule', 'forced_label', 'label'],
        'payload': (
            '...'
        ),
    },
    'V3_OOF': {
        'sha256': 'ec8135f36d6f634ac35b7574ecce5ae5ecc7e99bfe2509e72631625b14f18409',
        'rows': 14397,
        'columns': ['id', 'label', 'blend_prob'],
        'payload': (
            '...'
        ),
    },
    'V3_TEST': {
        'sha256': '31c4d30e41a137f98bb28b4dd4ccdb7f7ab51c349b5ea7a407391a9a52637e3f',
        'rows': 3603,
        'columns': ['id', 'blend_prob', 'label'],
        'payload': (
            '...'
        ),
    },
}


def load_frozen_csv(name):
    spec = FROZEN_ARTIFACTS[name]
    raw = gzip.decompress(base64.b64decode(spec["payload"]))
    digest = hashlib.sha256(raw).hexdigest()
    if digest != spec["sha256"]:
        raise RuntimeError(f"Frozen artifact checksum mismatch for {name}: {digest}")
    df = pd.read_csv(io.BytesIO(raw))
    if len(df) != spec["rows"] or df.columns.tolist() != spec["columns"]:
        raise RuntimeError(f"Frozen artifact schema mismatch for {name}")
    print(f"Loaded {name}: {df.shape} sha256={digest[:12]}...")
    return df

v2_oof = load_frozen_csv("V2_OOF")
v2_test = load_frozen_csv("V2_TEST")
v3_oof = load_frozen_csv("V3_OOF")
v3_test = load_frozen_csv("V3_TEST")


Loaded V2_OOF: (14397, 4) sha256=ed13cfd31361...
Loaded V2_TEST: (3603, 6) sha256=5f52f3664cf8...
Loaded V3_OOF: (14397, 3) sha256=ec8135f36d6f...
Loaded V3_TEST: (3603, 3) sha256=31c4d30e41a1...


In [7]:
def find_competition_csv(filename):
    roots = [Path("/kaggle/input"), Path("/mnt/data/ifest_dac")]
    candidates = []
    for root in roots:
        if not root.exists():
            continue
        candidates.extend(root.rglob(filename))
    if not candidates:
        raise FileNotFoundError(f"Could not find {filename} under Kaggle input")
    # Prefer the competition dataset path when present.
    candidates = sorted(candidates, key=lambda p: ("penyisihan-ifest-2026-dac" not in str(p).lower(), len(str(p))))
    return candidates[0]


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

train_path = find_competition_csv("train.csv")
test_path = find_competition_csv("test.csv")
sample_path = find_competition_csv("sample_submission.csv")

dataset_hash_mismatches = {}
for name, path in [("train", train_path), ("test", test_path), ("sample_submission", sample_path)]:
    digest = sha256_file(path)
    print(name, path, digest)
    if digest != EXPECTED_DATA_HASHES[name]:
        dataset_hash_mismatches[name] = {"expected": EXPECTED_DATA_HASHES[name], "actual": digest}

# ponytail: Kaggle mirrors may repackage identical competition data.
# Artifact alignment, V2 graph parity, and the final V4-change invariant below remain hard guards.
if dataset_hash_mismatches:
    print(f"WARNING: Dataset file hashes differ from the frozen snapshot: {dataset_hash_mismatches}")

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sample = pd.read_csv(sample_path)
assert train.shape[0] == EXPECTED_TRAIN_ROWS and test.shape[0] == EXPECTED_TEST_ROWS
assert train.columns.tolist() == ["id", "title", "content", "label"]
assert test.columns.tolist() == ["id", "title", "content"]
assert sample.columns.tolist() == ["id", "label"]
assert sample.id.astype(str).tolist() == test.id.astype(str).tolist()
assert set(train.label.unique()) == {0, 1}

# Frozen artifact alignment is mandatory.
assert v2_oof.id.astype(str).tolist() == train.id.astype(str).tolist()
assert v3_oof.id.astype(str).tolist() == train.id.astype(str).tolist()
assert v2_test.id.astype(str).tolist() == test.id.astype(str).tolist()
assert v3_test.id.astype(str).tolist() == test.id.astype(str).tolist()
assert np.array_equal(v2_oof.label.to_numpy(dtype=int), train.label.to_numpy(dtype=int))
assert np.array_equal(v3_oof.label.to_numpy(dtype=int), train.label.to_numpy(dtype=int))

print("Dataset and frozen-artifact alignment: OK")
print("train", train.shape, "test", test.shape)
print("train label counts", train.label.value_counts().sort_index().to_dict())

train /kaggle/input/competitions/penyisihan-ifest-2026-dac/train.csv a4ba91606302455da8a9523906a25361d581ac15b108964cdb3ab6beb0adb5d8
test /kaggle/input/competitions/penyisihan-ifest-2026-dac/test.csv 8e5de5bf53a50df87270dccdaba5ed0700d3729b5b03e4ce7724ac1769177005
sample_submission /kaggle/input/competitions/penyisihan-ifest-2026-dac/sample_submission.csv ea09389db317450c7b93da76d87b771daca79e66716b84b7432f1653a67582c6
Dataset and frozen-artifact alignment: OK
train (14397, 4) test (3603, 3)
train label counts {0: 1437, 1: 12960}


## Exact normalization and frozen V2 graph parity

In [8]:
TOKEN_RE = re.compile(r"[\w]+", flags=re.UNICODE)
SOURCE_SUFFIX_RE = re.compile(r"\s+kumparan\s+com\s*$", re.I)


def clean_ws(text):
    text = unicodedata.normalize("NFKC", str(text)).replace("\xa0", " ")
    return re.sub(r"\s+", " ", text).strip()


def clean_title_for_model(text):
    return SOURCE_SUFFIX_RE.sub("", clean_ws(text)).strip()


def normalize_key(text):
    return " ".join(TOKEN_RE.findall(clean_ws(text).lower()))

for df in (train, test):
    df["title_model"] = df.title.map(clean_title_for_model)
    df["title_key"] = df.title_model.map(normalize_key)
    df["content_key"] = df.content.map(normalize_key)
    df["pair_key"] = df.title_key + "|||" + df.content_key
    df["content_words"] = df.content.map(lambda x: len(clean_ws(x).split()))


def build_graph_reference(ref_df):
    pair_to_label = {}
    for key, g in ref_df.groupby("pair_key", sort=False):
        labs = set(map(int, g.label))
        if len(labs) == 1:
            pair_to_label[key] = next(iter(labs))
    content_to_pos_titles = defaultdict(set)
    title_to_pos_contents = defaultdict(set)
    for r in ref_df.loc[ref_df.label == 1].itertuples():
        content_to_pos_titles[r.content_key].add(r.title_key)
        title_to_pos_contents[r.title_key].add(r.content_key)
    return pair_to_label, content_to_pos_titles, title_to_pos_contents


def frozen_v2_graph_decisions(query_df, ref_df, v2_meta_prob):
    query_df = query_df.reset_index(drop=True)
    pair_to_label, content_to_pos_titles, title_to_pos_contents = build_graph_reference(ref_df)
    rules = np.array(["semantic"] * len(query_df), dtype=object)
    forced = np.full(len(query_df), -1, dtype=np.int8)
    v2_meta_prob = np.asarray(v2_meta_prob, dtype=float)
    if len(v2_meta_prob) != len(query_df):
        raise ValueError("v2_meta_prob/query length mismatch")

    for pos, r in enumerate(query_df.itertuples()):
        if r.pair_key in pair_to_label:
            forced[pos] = int(pair_to_label[r.pair_key])
            rules[pos] = "exact_pair"
            continue

        malformed = int(r.content_words) < MIN_GRAPH_CONTENT_WORDS
        pos_titles = content_to_pos_titles.get(r.content_key, set())
        if not malformed and len(pos_titles) == 1 and r.title_key not in pos_titles:
            forced[pos] = 0
            rules[pos] = "unique_content_different_title"
            continue

        pos_contents = title_to_pos_contents.get(r.title_key, set())
        if (
            not malformed and pos_contents and r.content_key not in pos_contents
            and v2_meta_prob[pos] < V2_TITLE_GATE
        ):
            forced[pos] = 0
            rules[pos] = "gated_title_different_content"

    return rules, forced

v2_test_meta = v2_test.meta_prob.to_numpy(dtype=float)
reconstructed_v2_rule, reconstructed_v2_forced = frozen_v2_graph_decisions(test, train, v2_test_meta)
assert np.array_equal(reconstructed_v2_rule, v2_test.graph_rule.astype(str).to_numpy()), "Frozen V2 graph rule parity failure"
assert np.array_equal(reconstructed_v2_forced, v2_test.forced_label.to_numpy(dtype=np.int8)), "Frozen V2 forced-label parity failure"
print("Frozen V2 graph parity: EXACT MATCH")
display(pd.Series(reconstructed_v2_rule).value_counts().to_frame("rows"))

Frozen V2 graph parity: EXACT MATCH


,rows
semantic,2447
exact_pair,927
unique_content_different_title,212
gated_title_different_content,17


## Structural profile that justifies the transductive constraint

In [9]:
def structural_profile(df):
    rows = []
    # Same content, multiple distinct titles.
    for content_key, g in df.groupby("content_key", sort=False):
        n_titles = g.title_key.nunique()
        if n_titles < 2:
            continue
        pos_titles = g.loc[g.label == 1, "title_key"].nunique()
        rows.append({"axis": "content", "group_key": content_key, "candidates": int(n_titles), "positive_candidates": int(pos_titles), "rows": len(g)})
    # Same title, multiple distinct contents.
    for title_key, g in df.groupby("title_key", sort=False):
        n_contents = g.content_key.nunique()
        if n_contents < 2:
            continue
        pos_contents = g.loc[g.label == 1, "content_key"].nunique()
        rows.append({"axis": "title", "group_key": title_key, "candidates": int(n_contents), "positive_candidates": int(pos_contents), "rows": len(g)})
    return pd.DataFrame(rows)

profile = structural_profile(train)
profile_summary = (
    profile.groupby("axis")
    .agg(groups=("group_key", "size"), exactly_one_positive=("positive_candidates", lambda s: int((s == 1).sum())), zero_positive=("positive_candidates", lambda s: int((s == 0).sum())), multi_positive=("positive_candidates", lambda s: int((s > 1).sum())))
    .reset_index()
)
profile_summary["exactly_one_positive_rate"] = profile_summary.exactly_one_positive / profile_summary.groups
print("Train structural profile")
display(profile_summary)
content_rate = float(profile_summary.loc[profile_summary.axis == "content", "exactly_one_positive_rate"].iloc[0])
assert content_rate >= 0.97, f"Same-content canonical prior unexpectedly weakened: {content_rate:.4f}"

Train structural profile


,axis,groups,exactly_one_positive,zero_positive,multi_positive,exactly_one_positive_rate
0,content,893,873,16,4,0.977604
1,title,121,117,0,4,0.966942


## Frozen score construction and V4 transductive package

In [10]:
# V2 remains the final-classification anchor. V3/global scores are used only for relative candidate ranking.
v2_oof_meta = v2_oof.meta_prob.to_numpy(dtype=float)
v2_test_meta = v2_test.meta_prob.to_numpy(dtype=float)
v2_oof_global = v2_oof.global_e.to_numpy(dtype=float)
v2_test_global = v2_test.global_e.to_numpy(dtype=float)
v3_oof_blend = v3_oof.blend_prob.to_numpy(dtype=float)
v3_test_blend = v3_test.blend_prob.to_numpy(dtype=float)

rank_oof = RANK_V3_WEIGHT * v3_oof_blend + RANK_V2_GLOBAL_WEIGHT * v2_oof_global
rank_test = RANK_V3_WEIGHT * v3_test_blend + RANK_V2_GLOBAL_WEIGHT * v2_test_global
assert np.isfinite(rank_oof).all() and np.isfinite(rank_test).all()
assert ((rank_oof >= 0) & (rank_oof <= 1)).all()
assert ((rank_test >= 0) & (rank_test <= 1)).all()

# Recover the frozen V2 semantic threshold only for validation reconstruction.
# Actual test labels are read directly from the already-submitted V2 artifact.
def exact_macro_f1_threshold(y, probs):
    y = np.asarray(y, dtype=np.int8)
    p = np.asarray(probs, dtype=float)
    order = np.argsort(p, kind="mergesort")
    ps = p[order]; ys = y[order]
    unique_vals, first_idx = np.unique(ps, return_index=True)
    candidates = [np.nextafter(float(ps.min()), -np.inf)]
    candidates.extend(float(x) for x in unique_vals)
    candidates.append(np.nextafter(float(ps.max()), np.inf))
    best_t, best_score = 0.5, -1.0
    for t in candidates:
        pred = (p >= t).astype(np.int8)
        score = f1_score(y, pred, average="macro", zero_division=0)
        if score > best_score + 1e-15 or (abs(score - best_score) <= 1e-15 and abs(t - 0.5) < abs(best_t - 0.5)):
            best_t, best_score = float(t), float(score)
    return best_t, best_score

V2_META_THRESHOLD, v2_oof_macro = exact_macro_f1_threshold(train.label.to_numpy(dtype=np.int8), v2_oof_meta)
print(f"Frozen V2 meta threshold recovered from OOF: {V2_META_THRESHOLD:.9f} | macro_f1={v2_oof_macro:.6f}")


def apply_transductive_v4(query_df, base_labels, base_forced, rank_scores, ref_df):
    """Apply only the pre-agreed V4 package. V2-forced rows are immutable."""
    q = query_df.reset_index(drop=True).copy()
    base_labels = np.asarray(base_labels, dtype=np.int8)
    base_forced = np.asarray(base_forced, dtype=np.int8)
    rank_scores = np.asarray(rank_scores, dtype=float)
    n = len(q)
    if not (len(base_labels) == len(base_forced) == len(rank_scores) == n):
        raise ValueError("V4 input length mismatch")

    _, content_to_pos_titles, title_to_pos_contents = build_graph_reference(ref_df)
    labels = base_labels.copy()
    v4_forced = np.full(n, -1, dtype=np.int8)
    v4_rule = np.array(["v2_fallback"] * n, dtype=object)
    locked = base_forced >= 0
    v4_rule[locked] = "v2_graph_locked"
    group_records = []

    # -------- 1) SAME CONTENT / DIFFERENT TITLES --------
    available = np.where(~locked)[0]
    if len(available):
        aq = q.iloc[available].copy()
        aq["_pos"] = available
        for content_key, g in aq.groupby("content_key", sort=False):
            if int(g.content_words.iloc[0]) < MIN_GRAPH_CONTENT_WORDS:
                continue
            if g.title_key.nunique() < 2:
                continue
            # V4 acts only when train does not already identify any positive title for this body.
            if len(content_to_pos_titles.get(content_key, set())) > 0:
                continue

            cand = (
                g.assign(_score=rank_scores[g._pos.to_numpy(dtype=int)])
                .groupby("title_key", as_index=False)
                .agg(score=("_score", "mean"), rows=("_pos", "size"))
                .sort_values(["score", "title_key"], ascending=[False, True], kind="mergesort")
                .reset_index(drop=True)
            )
            top_score = float(cand.score.iloc[0])
            if top_score < CONTENT_CANONICAL_MIN_SCORE:
                group_records.append({"axis":"content", "group_key":content_key, "candidates":len(cand), "top_score":top_score, "margin":np.nan, "applied":False, "top_key":cand.title_key.iloc[0]})
                continue
            top_key = cand.title_key.iloc[0]
            pos_idx = g._pos.to_numpy(dtype=int)
            new = (g.title_key.to_numpy() == top_key).astype(np.int8)
            labels[pos_idx] = new
            v4_forced[pos_idx] = new
            v4_rule[pos_idx] = "test_content_canonical"
            locked[pos_idx] = True
            group_records.append({"axis":"content", "group_key":content_key, "candidates":len(cand), "top_score":top_score, "margin":np.nan, "applied":True, "top_key":top_key})

    # -------- 2) SAME TITLE / DIFFERENT CONTENTS --------
    available = np.where(~locked)[0]
    if len(available):
        aq = q.iloc[available].copy()
        aq["_pos"] = available
        for title_key, g in aq.groupby("title_key", sort=False):
            if g.content_key.nunique() < 2:
                continue
            # If train knows a positive content for this title, V2 train->test graph owns the decision.
            if len(title_to_pos_contents.get(title_key, set())) > 0:
                continue

            cand = (
                g.assign(_score=rank_scores[g._pos.to_numpy(dtype=int)])
                .groupby("content_key", as_index=False)
                .agg(score=("_score", "mean"), rows=("_pos", "size"))
                .sort_values(["score", "content_key"], ascending=[False, True], kind="mergesort")
                .reset_index(drop=True)
            )
            top_score = float(cand.score.iloc[0])
            second_score = float(cand.score.iloc[1])
            margin = top_score - second_score
            if top_score < TITLE_CANONICAL_MIN_SCORE or margin < TITLE_CANONICAL_MIN_MARGIN:
                group_records.append({"axis":"title", "group_key":title_key, "candidates":len(cand), "top_score":top_score, "margin":margin, "applied":False, "top_key":cand.content_key.iloc[0]})
                continue
            top_key = cand.content_key.iloc[0]
            pos_idx = g._pos.to_numpy(dtype=int)
            new = (g.content_key.to_numpy() == top_key).astype(np.int8)
            labels[pos_idx] = new
            v4_forced[pos_idx] = new
            v4_rule[pos_idx] = "test_title_canonical"
            locked[pos_idx] = True
            group_records.append({"axis":"title", "group_key":title_key, "candidates":len(cand), "top_score":top_score, "margin":margin, "applied":True, "top_key":top_key})

    # Hard invariant: V4 must never override a V2 graph-forced row.
    immutable = base_forced >= 0
    if not np.array_equal(labels[immutable], base_labels[immutable]):
        raise RuntimeError("V4 attempted to override frozen V2 graph decisions")
    return labels, v4_rule, v4_forced, pd.DataFrame(group_records)

Frozen V2 meta threshold recovered from OOF: 0.596982825 | macro_f1=0.836466


## Validation A — component-disjoint transductive folds

In [11]:
class UnionFind:
    def __init__(self, n):
        self.p = list(range(n)); self.r = [0] * n
    def find(self, x):
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]; x = self.p[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb: return
        if self.r[ra] < self.r[rb]: ra, rb = rb, ra
        self.p[rb] = ra
        if self.r[ra] == self.r[rb]: self.r[ra] += 1


def build_component_groups(df):
    uf = UnionFind(len(df))
    for col in ["title_key", "content_key"]:
        first = {}
        for i, key in enumerate(df[col].tolist()):
            if key in first: uf.union(i, first[key])
            else: first[key] = i
    roots = [uf.find(i) for i in range(len(df))]
    mapping = {r:j for j,r in enumerate(dict.fromkeys(roots))}
    return np.asarray([mapping[r] for r in roots], dtype=np.int32)


def base_v2_for_query(query_df, ref_df, meta_prob):
    semantic = (np.asarray(meta_prob, dtype=float) >= V2_META_THRESHOLD).astype(np.int8)
    rules, forced = frozen_v2_graph_decisions(query_df, ref_df, meta_prob)
    labels = semantic.copy()
    m = forced >= 0
    labels[m] = forced[m]
    return labels, rules, forced

groups = build_component_groups(train)
cv = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_rows = []
for fold, (tr_idx, va_idx) in enumerate(cv.split(train, train.label.values, groups=groups)):
    assert set(groups[tr_idx]).isdisjoint(set(groups[va_idx]))
    ref = train.iloc[tr_idx].reset_index(drop=True)
    qry = train.iloc[va_idx].reset_index(drop=True)
    yq = train.label.to_numpy(dtype=np.int8)[va_idx]
    base, _, forced = base_v2_for_query(qry, ref, v2_oof_meta[va_idx])
    v4, _, _, _ = apply_transductive_v4(qry, base, forced, rank_oof[va_idx], ref)
    base_f1 = f1_score(yq, base, average="macro", zero_division=0)
    v4_f1 = f1_score(yq, v4, average="macro", zero_division=0)
    f0, f1 = f1_score(yq, v4, labels=[0,1], average=None, zero_division=0)
    fold_rows.append({"fold":fold, "rows":len(va_idx), "base_v2_macro_f1":base_f1, "v4_transductive_macro_f1":v4_f1, "gain":v4_f1-base_f1, "v4_f1_0":f0, "v4_f1_1":f1, "changes":int((v4 != base).sum())})

component_metrics = pd.DataFrame(fold_rows)
print("Component-disjoint transductive validation")
display(component_metrics)
print("Mean base:", component_metrics.base_v2_macro_f1.mean())
print("Mean V4  :", component_metrics.v4_transductive_macro_f1.mean())

Component-disjoint transductive validation


,fold,rows,base_v2_macro_f1,v4_transductive_macro_f1,gain,v4_f1_0,v4_f1_1,changes
0,0,4798,0.847747,0.885052,0.037305,0.794995,0.975108,85
1,1,4807,0.814714,0.890470,0.075755,0.802905,0.978035,129
2,2,4792,0.843990,0.896858,0.052868,0.817006,0.976711,110


Mean base: 0.8354837006687358
Mean V4  : 0.8907932400047347


## Validation B — 100 repeated pseudo-competition splits

In [12]:
y_all = train.label.to_numpy(dtype=np.int8)
idx_all = np.arange(len(train))
pseudo_rows = []
for k in range(N_PSEUDO_SPLITS):
    tr_idx, q_idx = train_test_split(idx_all, test_size=0.20, random_state=SEED + k, stratify=y_all)
    ref = train.iloc[tr_idx].reset_index(drop=True)
    qry = train.iloc[q_idx].reset_index(drop=True)
    base, _, forced = base_v2_for_query(qry, ref, v2_oof_meta[q_idx])
    v4, _, _, _ = apply_transductive_v4(qry, base, forced, rank_oof[q_idx], ref)
    yq = y_all[q_idx]
    base_f1 = f1_score(yq, base, average="macro", zero_division=0)
    v4_f1 = f1_score(yq, v4, average="macro", zero_division=0)
    pseudo_rows.append({"split":k, "base_v2_macro_f1":base_f1, "v4_macro_f1":v4_f1, "gain":v4_f1-base_f1, "changes":int((v4 != base).sum())})

pseudo_metrics = pd.DataFrame(pseudo_rows)
wins = int((pseudo_metrics.gain > 1e-12).sum())
ties = int((pseudo_metrics.gain.abs() <= 1e-12).sum())
losses = int((pseudo_metrics.gain < -1e-12).sum())
print(f"Pseudo-competition: wins={wins} ties={ties} losses={losses} / {N_PSEUDO_SPLITS}")
print("Mean base:", pseudo_metrics.base_v2_macro_f1.mean())
print("Mean V4  :", pseudo_metrics.v4_macro_f1.mean())
print("Mean gain:", pseudo_metrics.gain.mean())
display(pseudo_metrics.describe())
# The agreed V4 direction was accepted only because repeated pseudo-test support was extremely stable.
assert wins >= 95, f"V4 pseudo-competition stability failed: only {wins}/{N_PSEUDO_SPLITS} wins"

Pseudo-competition: wins=100 ties=0 losses=0 / 100
Mean base: 0.9103778341719
Mean V4  : 0.918311008625545
Mean gain: 0.007933174453645178


,split,base_v2_macro_f1,v4_macro_f1,gain,changes
count,100.000000,100.000000,100.000000,100.000000,100.000000
mean,49.500000,0.910378,0.918311,0.007933,9.930000
std,29.011492,0.008172,0.007501,0.003513,3.461929
min,0.000000,0.892724,0.902187,0.001266,4.000000
25%,24.750000,0.904886,0.912949,0.005326,7.000000
50%,49.500000,0.909634,0.917413,0.007934,10.000000
75%,74.250000,0.916736,0.922904,0.009932,12.000000
max,99.000000,0.930548,0.937766,0.016551,20.000000


## V4 test inference — frozen V2 base + test-test canonical matching

In [13]:
base_v2_label = v2_test.label.to_numpy(dtype=np.int8)
base_v2_forced = v2_test.forced_label.to_numpy(dtype=np.int8)

final_v4_label, v4_rule, v4_forced, canonical_groups = apply_transductive_v4(
    test, base_v2_label, base_v2_forced, rank_test, train
)

changes_vs_v2 = int((final_v4_label != base_v2_label).sum())
print("V2 distribution:", pd.Series(base_v2_label).value_counts().sort_index().to_dict())
print("V4 distribution:", pd.Series(final_v4_label).value_counts().sort_index().to_dict())
print("V4 changes vs V2:", changes_vs_v2)
print("Applied V4 rules:")
display(pd.Series(v4_rule).value_counts().to_frame("rows"))
print("Canonical group decisions:")
display(canonical_groups.groupby(["axis", "applied"]).size().to_frame("groups"))

# Methodology-drift guard from the pre-implementation audit.
assert changes_vs_v2 == EXPECTED_V4_CHANGES_VS_V2, f"Expected {EXPECTED_V4_CHANGES_VS_V2} targeted changes, got {changes_vs_v2}; aborting to prevent methodology drift"
assert np.array_equal(final_v4_label[base_v2_forced >= 0], base_v2_label[base_v2_forced >= 0])

change_mask = final_v4_label != base_v2_label
changed = pd.DataFrame({
    "id": test.id,
    "title": test.title,
    "v2_meta_prob": v2_test_meta,
    "v3_blend_prob": v3_test_blend,
    "v2_global_e": v2_test_global,
    "rank_score": rank_test,
    "v2_graph_rule": v2_test.graph_rule,
    "v2_forced_label": base_v2_forced,
    "v4_rule": v4_rule,
    "v4_forced_label": v4_forced,
    "v2_label": base_v2_label,
    "v4_label": final_v4_label,
}).loc[change_mask].reset_index(drop=True)
display(changed)

V2 distribution: {0: 369, 1: 3234}
V4 distribution: {0: 383, 1: 3220}
V4 changes vs V2: 16
Applied V4 rules:


,rows
v2_fallback,2369
v2_graph_locked,1156
test_content_canonical,78


Canonical group decisions:


,,groups
axis,applied,
content,True,37


,id,title,v2_meta_prob,v3_blend_prob,v2_global_e,rank_score,v2_graph_rule,v2_forced_label,v4_rule,v4_forced_label,v2_label,v4_label
0,te00062,Ribuan Buruh Cianjur Kepung Pendopo Massa Abaikan Prokes,0.959601,0.934720,0.925364,0.929574,semantic,-1,test_content_canonical,0,1,0
1,te00327,TPU Pondok Ranggon Hampir Penuh Sisa 15 Liang Makam untuk Jenazah Corona kumparan com,0.870973,0.917091,0.627629,0.757887,semantic,-1,test_content_canonical,0,1,0
2,te00389,PSBB Transisi Anies dan Lonjakan 2 607 Kasus Corona Jakarta,0.822901,0.850521,0.864365,0.858136,semantic,-1,test_content_canonical,0,1,0
3,te00506,Batik Air Bawa 5 Penumpang Positif COVID Penumpang Lain Bisa Tertular,0.985494,0.976244,0.987134,0.982234,semantic,-1,test_content_canonical,0,1,0
4,te00558,Kapolda Kalteng Terima Kasih Sudah Terapkan Prokes Saat di TPS,0.568754,0.696866,0.994969,0.860823,semantic,-1,test_content_canonical,1,0,1
5,te00933,6 500 Alat Cek Corona GeNose Rp20 Ribu Disebar Februari 2021,0.923913,0.943976,0.967663,0.957004,semantic,-1,test_content_canonical,0,1,0
6,te00991,Video Anji soal Obat Corona Slamet Budiarto di Youtube Hilang,0.762117,0.850400,0.964797,0.913319,semantic,-1,test_content_canonical,0,1,0
7,te01239,Hoaks Larangan ke Kota Lamongan Karena Zona Hitam Disebar di Warung Kopi,0.777672,0.831692,0.801808,0.815256,semantic,-1,test_content_canonical,0,1,0
8,te01448,Sidang Tahunan DPR Kehadiran Fisik Pejabat Publik Dibatasi,0.967752,0.961645,0.737329,0.838271,semantic,-1,test_content_canonical,0,1,0
9,te01808,Wagub Sulsel Batal Divaksin karena Kerabat Positif COVID Bagaimana Nakes,0.976373,0.956859,0.998970,0.980020,semantic,-1,test_content_canonical,0,1,0


In [14]:
# ---------------- EXPORT ----------------
submission = sample[["id"]].copy()
submission["label"] = final_v4_label.astype(int)

submission_path = OUTPUT_DIR / "submission_v4.csv"
test_scores_path = OUTPUT_DIR / "test_scores_v4.csv"
profile_path = OUTPUT_DIR / "structural_profile_v4.csv"
component_path = OUTPUT_DIR / "component_transductive_metrics_v4.csv"
pseudo_path = OUTPUT_DIR / "pseudo_competition_metrics_v4.csv"
groups_path = OUTPUT_DIR / "canonical_groups_v4.csv"
changes_path = OUTPUT_DIR / "changed_rows_v4.csv"
config_path = OUTPUT_DIR / "config_v4.json"
zip_path = OUTPUT_DIR / "results_v4.zip"

submission.to_csv(submission_path, index=False)
pd.DataFrame({
    "id": test.id,
    "v2_meta_prob": v2_test_meta,
    "v3_blend_prob": v3_test_blend,
    "v2_global_e": v2_test_global,
    "rank_score": rank_test,
    "v2_graph_rule": v2_test.graph_rule.astype(str),
    "v2_forced_label": base_v2_forced.astype(int),
    "v2_label": base_v2_label.astype(int),
    "v4_rule": v4_rule,
    "v4_forced_label": v4_forced.astype(int),
    "label": final_v4_label.astype(int),
}).to_csv(test_scores_path, index=False)
profile.to_csv(profile_path, index=False)
component_metrics.to_csv(component_path, index=False)
pseudo_metrics.to_csv(pseudo_path, index=False)
canonical_groups.to_csv(groups_path, index=False)
changed.to_csv(changes_path, index=False)

config = {
    "version": "v4-transductive-canonical-matching",
    "training": False,
    "seed": SEED,
    "base_system": "frozen-v2",
    "rank": {"v3_blend_weight": RANK_V3_WEIGHT, "v2_global_weight": RANK_V2_GLOBAL_WEIGHT},
    "content_rule": {"min_score": CONTENT_CANONICAL_MIN_SCORE, "min_content_words": MIN_GRAPH_CONTENT_WORDS},
    "title_rule": {"min_score": TITLE_CANONICAL_MIN_SCORE, "min_margin": TITLE_CANONICAL_MIN_MARGIN},
    "v2_title_gate": V2_TITLE_GATE,
    "pseudo_splits": N_PSEUDO_SPLITS,
    "pseudo_wins": int((pseudo_metrics.gain > 1e-12).sum()),
    "pseudo_losses": int((pseudo_metrics.gain < -1e-12).sum()),
    "pseudo_mean_gain": float(pseudo_metrics.gain.mean()),
    "component_mean_gain": float(component_metrics.gain.mean()),
    "changes_vs_v2": changes_vs_v2,
    "test_negative_count": int((final_v4_label == 0).sum()),
    "public_lb_reference": {"v1": PUBLIC_LB_V1, "v2": PUBLIC_LB_V2, "v3": PUBLIC_LB_V3},
}
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

files = [submission_path, test_scores_path, profile_path, component_path, pseudo_path, groups_path, changes_path, config_path]
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in files:
        zf.write(p, arcname=p.name)

# Final submission integrity.
check = pd.read_csv(submission_path)
assert check.shape == (EXPECTED_TEST_ROWS, 2)
assert check.columns.tolist() == ["id", "label"]
assert check.id.astype(str).tolist() == test.id.astype(str).tolist()
assert set(check.label.unique()).issubset({0,1})
assert not check.isna().any().any()

print("Saved V4 artifacts:")
for p in files + [zip_path]:
    print(" -", p)
print("\nSUBMISSION READY:", submission_path)
print("Changes vs V2:", changes_vs_v2)
print("Final label distribution:", check.label.value_counts().sort_index().to_dict())

Saved V4 artifacts:
 - /kaggle/working/submission_v4.csv
 - /kaggle/working/test_scores_v4.csv
 - /kaggle/working/structural_profile_v4.csv
 - /kaggle/working/component_transductive_metrics_v4.csv
 - /kaggle/working/pseudo_competition_metrics_v4.csv
 - /kaggle/working/canonical_groups_v4.csv
 - /kaggle/working/changed_rows_v4.csv
 - /kaggle/working/config_v4.json
 - /kaggle/working/results_v4.zip

SUBMISSION READY: /kaggle/working/submission_v4.csv
Changes vs V2: 16
Final label distribution: {0: 383, 1: 3220}
